<a href="https://colab.research.google.com/github/whynotkavi/self-care-ai/blob/main/notebooks/01_emotion_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


In [ ]:
!pip install -q transformers datasets accelerate scikit-learn


In [ ]:
from datasets import load_dataset

data = load_dataset("google-research-datasets/go_emotions", "simplified")
print(data)
print(data["train"][0])


README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

simplified/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

simplified/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  347kB            

simplified/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
{'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej'}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset
data = load_dataset("google-research-datasets/go_emotions", "simplified")
names = data["train"].features["labels"].feature.names
print(data)
print(names)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [ ]:
groups = {
    "happy":     ["amusement","joy","love","gratitude","relief","caring"],
    "sad":       ["sadness","grief","disappointment","remorse","embarrassment"],
    "stressed":  ["fear","nervousness","confusion"],
    "angry":     ["anger","annoyance","disgust","disapproval"],
    "motivated": ["excitement","optimism","pride","desire"],
    "neutral":   ["neutral"],
}
name_to_group = {n: g for g, ns in groups.items() for n in ns}
my_labels = list(groups.keys())
label2id = {l: i for i, l in enumerate(my_labels)}
id2label = {i: l for l, i in label2id.items()}

def convert(ex):
    ids = ex["labels"]
    if len(ids) != 1:
        return {"label": -1}
    g = name_to_group.get(names[ids[0]])
    return {"label": label2id[g] if g else -1}

data = data.map(convert, remove_columns=["labels", "id"])
data = data.filter(lambda x: x["label"] != -1)
print({k: len(v) for k, v in data.items()})

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Filter:   0%|          | 0/43410 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5426 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5427 [00:00<?, ? examples/s]

{'train': 29030, 'validation': 3631, 'test': 3654}


In [ ]:
import random
train = data["train"]
neutral_id = label2id["neutral"]
idx_n = [i for i, l in enumerate(train["label"]) if l == neutral_id]
idx_o = [i for i, l in enumerate(train["label"]) if l != neutral_id]
random.seed(42)
keep = idx_o + random.sample(idx_n, min(4000, len(idx_n)))
data["train"] = train.select(sorted(keep))

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
data = data.map(lambda b: tok(b["text"], truncation=True, max_length=64), batched=True)
print({k: len(v) for k, v in data.items()})

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/20207 [00:00<?, ? examples/s]

Map:   0%|          | 0/3631 [00:00<?, ? examples/s]

Map:   0%|          | 0/3654 [00:00<?, ? examples/s]

{'train': 20207, 'validation': 3631, 'test': 3654}


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(my_labels),
    id2label=id2label, label2id=label2id)

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "f1": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir="/content/out", eval_strategy="epoch", save_strategy="no",
    learning_rate=2e-5, per_device_train_batch_size=32,
    per_device_eval_batch_size=64, num_train_epochs=2,
    weight_decay=0.01, fp16=True, report_to="none")

trainer = Trainer(model=model, args=args,
    train_dataset=data["train"], eval_dataset=data["validation"],
    processing_class=tok, data_collator=DataCollatorWithPadding(tok),
    compute_metrics=metrics)
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.015678,0.878708,0.672817,0.627316
2,0.746170,0.852099,0.677775,0.641645


TrainOutput(global_step=1264, training_loss=0.8411437650270099, metrics={'train_runtime': 59.7242, 'train_samples_per_second': 676.677, 'train_steps_per_second': 21.164, 'total_flos': 378746651408760.0, 'train_loss': 0.8411437650270099, 'epoch': 2.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
save_path = "/content/drive/MyDrive/self-care-ai/emotion_model"
os.makedirs(save_path, exist_ok=True)
trainer.save_model(save_path)
tok.save_pretrained(save_path)
print("Saved to", save_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/self-care-ai/emotion_model


In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model=save_path, device=0)
tests = [
    "I have three deadlines and I can't breathe",
    "I finally finished my project, I feel amazing",
    "I'm so sick of everything going wrong",
    "Nobody texted me today",
    "I'm ready to work out and crush today",
]
for t in tests:
    print(t, "->", clf(t)[0])


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

I have three deadlines and I can't breathe -> {'label': 'sad', 'score': 0.5546450614929199}
I finally finished my project, I feel amazing -> {'label': 'happy', 'score': 0.6542434692382812}
I'm so sick of everything going wrong -> {'label': 'sad', 'score': 0.5981252789497375}
Nobody texted me today -> {'label': 'neutral', 'score': 0.5350629687309265}
I'm ready to work out and crush today -> {'label': 'motivated', 'score': 0.872079074382782}
